# Account ingestion: Raw to Landing to Bronze

##### Prajwol Regmi

In [0]:
%run ./02_utility

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Landing Layer

In [0]:
if batch_id in ("2", "3"):
    log_checkpoint ("account_landing_ingestion", "in_progress")

    account_schema = StructType([
        StructField("CDC_FLAG", StringType(), True),
        StructField("CDC_DSN", StringType(), True),
        StructField("CA_ID", StringType(), True),
        StructField("CA_C_ID", StringType(), True),
        StructField("CA_B_ID", StringType(), True),
        StructField("CA_NAME", StringType(), True),
        StructField("CA_TAX_ST", StringType(), True),
        StructField("CA_ST_ID", StringType(), True),
    ])

    df_account_raw = spark.read.option('delimiter', '|') \
        .option("header", "false") \
        .schema(account_schema) \
        .csv(f"{batch_path}/Account.txt")

    df_account_landing = df_account_raw \
        .withColumn("_landing_ts", current_timestamp()) \
        .withColumn("_batch", lit(batch_id)) \
        .withColumn("_source_file",lit("Account.txt")) \
        .withColumn("_run_id", lit(run_id))

    df_account_landing.write \
        .mode("overwrite") \
        .parquet(f"{landing_volume}/account")

    account_landing_count = df_account_raw.count()

    print(f"Account.txt landing count: {account_landing_count} rows {account_landing_path} ")
    log_audit("account", "landing_write", account_landing_count)


    df_account_bronze = spark.read.parquet(account_landing_path) \
        .drop("_landing_ts") \
        .withColumn("_ingest_ts", current_timestamp())

    # write bronze table

    df_account_bronze.write.mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{catalog}.{bronze_schema}.account")

    account_bronze_count = df_account_bronze.count()
    print(f" Account.txt bronze: {account_bronze_count} rows appended")
    log_audit("account", "bronze_append", account_bronze_count)

else:
    print("skipping account.txt, file not in batch1")


In [0]:
# logic for batch 1:
# schema has 4 fields

if batch_id == "1":
    ct_schema = StructType([
        StructField("CT_CA_ID", StringType(), True),
        StructField("CT_DTS", StringType(), True),
        StructField("CT_AMT", StringType(), True),
        StructField("CT_NAME", StringType(), True)
    ])

#in batch 2 and 3, cashtransaction has 6 fields
else:
    ct_schema = StructType([
        StructField("CDC_FLAG", StringType(), True),
        StructField("CDC_DSN", StringType(), True),
        StructField("CT_CA_ID", StringType(), True),
        StructField("CT_DTS", StringType(), True),
        StructField("CT_AMT", StringType(), True),
        StructField("CT_NAME", StringType(), True)
    ])

In [0]:
df_ct_raw = spark.read.option("delimiter", "|") \
    .option("header", "false") \
    .schema(ct_schema) \
    .csv(f"{batch_path}/CashTransaction.txt")

In [0]:
if batch_id == "1":
    df_ct_raw = (
        df_ct_raw.withColumn("CDC_FLAG", lit("I")) \
            .withColumn("CDC_DSN", lit(None).cast(StringType())) \
            .select("CDC_FLAG", "CDC_DSN", "CT_CA_ID", "CT_DTS", "CT_AMT", "CT_NAME")
    )


In [0]:
df_ct_landing = df_ct_raw.withColumn("_landing_ts", current_timestamp()) \
    .withColumn("_batch", lit(batch_id)) \
    .withColumn("_source_file", lit("CashTransaction.txt")) \
    .withColumn("_run_id", lit(run_id))

In [0]:
ct_landing_path = f"{landing_volume}/cashtransaction"
df_ct_landing.write.mode("overwrite").parquet(ct_landing_path)

ct_landing_count = df_ct_landing.count()
print(f"CashTransaction.txt landing count: {ct_landing_count} rows {ct_landing_path} ")
log_audit("cashtransaction", "landing_write", ct_landing_count)

# Bronze Layer

In [0]:
df_ct_bronze = spark.read.parquet(ct_landing_path) \
    .drop("_landing_ts") \
    .withColumn("_ingest_ts", current_timestamp())

df_ct_bronze.write.mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.{bronze_schema}.cashtransaction")

ct_bronze_count = df_ct_bronze.count()
print(f"Rows appended: {ct_bronze_count} ")
log_audit("cashtransaction", "bronze_append", ct_bronze_count)
log_checkpoint("cash_transaction_ingestion_landing", "completed", ct_bronze_count)

In [0]:
display(df_ct_bronze)